# Part 4: Keras Sequential, `compile` and `fit`

Keras is a high-level library. You describe the model, tell it the loss and optimizer with **`compile`**, and **`fit`** runs the whole training loop for you. This is the same model and the same data as the PyTorch notebooks, so you can compare.

In [13]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow start-up messages

import keras
from keras import layers

import numpy as np

from churn_data import make_splits

splits = make_splits(42)
x_train, y_train = splits.train.x.numpy(), splits.train.y.numpy()
x_val, y_val = splits.val.x.numpy(), splits.val.y.numpy()
print(x_train.shape, y_train.mean().round(3))

(1200, 9) 0.141


## Build, compile, fit

`keras.Input` states the shape of one row. Each `Dense` layer works out its own input size, so you only give it the number of outputs. The last layer has no activation, so it returns a **logit**, and the loss is told so with `from_logits=True`. This matches `BCEWithLogitsLoss` in PyTorch.

In [14]:
keras.utils.set_random_seed(42)

model = keras.Sequential(
    [
        keras.Input(shape=(9,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ],
    name="churn_net",
)
model.summary()

Model: "churn_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 16)             │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 449 (1.75 KB)

 Trainable params: 449 (1.75 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(
    optimizer=keras.optimizers.Adam(3e-3),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.AUC(from_logits=True, name="auc")],
)
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=30,
    batch_size=64,
    verbose=0,
)
print(list(history.history))
print("epoch 1: ", {k: round(v[0], 3) for k, v in history.history.items()})
print("epoch 30:", {k: round(v[-1], 3) for k, v in history.history.items()})

['auc', 'loss', 'val_auc', 'val_loss']
epoch 1:  {'auc': 0.555, 'loss': 0.482, 'val_auc': 0.654, 'val_loss': 0.399}
epoch 30: {'auc': 0.803, 'loss': 0.328, 'val_auc': 0.779, 'val_loss': 0.34}


`fit` returns a **History** object. `history.history` is a dictionary with one list per metric, one value per epoch, named `loss`, `val_loss` and so on. That is the same table you built by hand in Module 07. Notice what you did not write: no `zero_grad`, no `backward`, no `step`, no `model.train()`. `fit` does them.

## Evaluate and predict

In [16]:
print(model.evaluate(x_val, y_val, verbose=0, return_dict=True))

scores = model.predict(x_val, verbose=0)
print(scores.shape, scores[:3].ravel().round(2))
print("probabilities:", (1 / (1 + np.exp(-scores[:3].ravel()))).round(3))

{'auc': 0.7792514562606812, 'loss': 0.34048938751220703}
(400, 1) [-1.59 -4.52 -2.84]
probabilities: [0.169 0.011 0.055]


`predict` returns a NumPy array of shape `(N, 1)` holding logits, so call `.ravel()` before you pass it to scikit-learn, and apply the sigmoid yourself to get probabilities. Calling the model directly, `model(x)`, returns a tensor for one batch.

## The class imbalance, once more

About 14% of the customers churn. In PyTorch you passed `pos_weight` to the loss. In Keras the same idea is `class_weight` in `fit`.

In [17]:
from sklearn.metrics import f1_score

negatives = len(y_train) - y_train.sum()
weights = {0: 1.0, 1: float(negatives / y_train.sum())}
print("class_weight:", {k: round(v, 2) for k, v in weights.items()})


def fit_model(class_weight=None):
    keras.utils.set_random_seed(42)
    net = keras.Sequential(
        [keras.Input(shape=(9,)), layers.Dense(16, activation="relu"),
         layers.Dense(16, activation="relu"), layers.Dense(1)]
    )
    net.compile(optimizer=keras.optimizers.Adam(3e-3),
                loss=keras.losses.BinaryCrossentropy(from_logits=True))
    net.fit(x_train, y_train, epochs=30, batch_size=64, verbose=0, class_weight=class_weight)
    return net


for label, class_weight in (("no weights", None), ("class_weight", weights)):
    logits = fit_model(class_weight).predict(x_val, verbose=0).ravel()
    predicted = (logits >= 0).astype(int)
    print(f"{label:13s} validation F1 {f1_score(y_val, predicted):.3f}  predicted positives {predicted.sum()} of {int(y_val.sum())} real")

class_weight: {0: 1.0, 1: 6.1}
no weights    validation F1 0.222  predicted positives 16 of 56 real
class_weight  validation F1 0.398  predicted positives 150 of 56 real


Without weights the model plays safe and flags only 16 customers. With weights it flags 150, so it catches more real churners at the cost of many false alarms, and F1 rises from 0.22 to 0.40.

## PyTorch to Keras, side by side

| PyTorch | Keras |
|---|---|
| `nn.Sequential`, `nn.Linear(9, 16)` | `keras.Sequential`, `layers.Dense(16)` with `keras.Input(shape=(9,))` |
| `nn.BCEWithLogitsLoss()` | `BinaryCrossentropy(from_logits=True)` |
| `torch.optim.Adam(params, lr=3e-3)` | `keras.optimizers.Adam(3e-3)` |
| hand-written epoch and batch loops | `model.fit(..., epochs=, batch_size=)` |
| `pos_weight` in the loss | `class_weight` in `fit` |
| `model.eval()` and `no_grad()` | handled inside `predict` and `evaluate` |
| `torch.save(model.state_dict(), ...)` | `model.save("model.keras")` |

## Your turn

Change `batch_size` to 16, then 256, and compare the time and the final `val_loss`.